# CSE 5243 - Introduction to Data Mining
## Homework 5: Association Analysis
- Semester: Spring 2026
- Instructor: Dr. Thomas Bihari
- Section: WF, 9:35 - 10:55 a.m. 
- Student Name: Sayan Bisi
- Student Email: bisi.2@buckeyemail.osu.edu

Template Version V2.
***

# Introduction

### Objectives

In this lab, you will use a grocery dataset provided on Carmen to find potential association rules.

The objectives of this assignment are:
- Practice the Association Analysis content we covered this semester.
- Understand “why” the particular topics, techniques, etc., are important from a practical perspective.
- Understand how to choose and use appropriate tools to solve the provided problems.

### The Dataset
- This workbook uses a market basket dataset containing transactions.  See the Excel dataset file for details.
- It is based on the French Bakery Daily Sales dataset (https://www.kaggle.com/datasets/matthieugimbert/french-bakery-daily-sales).  However, it has been reformatted and modified specifically for this asignment.  Do not publish this dataset or use it for purposes other than this assignment.
  - NOTE: Read the information about the original dataset, and review the data for issues or  considerations you might want to address.  In particular, some "Quantity" values are negative.  What does that mean?  Also, some of the Items may not be actual products (e.g., "Slice" is a separate fee for slicing bread.)  What do you choose to do with those cases?  You may choose - but you must justify your choices.
  - NOTE: For simplicity, you may set all Quantities to 1, so only one of each Item is purchased in each transaction.
- The data file captures the data in "long format". Specifically, every row corresponds to the transaction id and the item. If the specific transaction id has multiple items, there will be multiple rows in the data.
- You can process the data however you like, but it is recommended you convert into an encoded data structure suitable for use with the mlxtend package.  **This has been done for you below.**

## The Business Problem
- Assume this dataset contains all of the transactions for one month for our store.  We wish to find association rules that would improve our revenue as follows:
  - We would discount **one** of our products by **10%** each month, with the hope that this would encourage customers to visit our store to purchase that product **8%** more frequently, and also purchase other products (that are not discounted) more frequently.
- Practically speaking, we would like to come up with **two-item** rules (one antecedent and one consequent: (A -> B)) and choose the one that best adds to our revenues  (based on the rule support, confidence, etc.).
- For simplicity, don't consider complex rule interactions (e.g., A->B and B->C, A->B and B->A, etc.).  Assume each rule is completely separate.

### Proper Answers
- **IMPORTANT:** **Show your work** and **explain it**.  This will help us give partial credit in some cases.

### Collaboration
For this assignment, you should work as an individual. You may informally discuss ideas with classmates, but your work should be your own.

### What You Need to Turn In
- Submit this Jupyter Notebook in .IPYNB format.  Do not "zip" the file.

### Notes
- Feel free to use the **mlxtend** package throughout this assignment.  There are useful examples here!
  - See: **https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/association_rules/#example-1-generating-association-rules-from-frequent-itemsets**
***

***
# Section: 1 - Get Ready
1A) Load the data, and get it ready for association analysis. Do this with convenient python helper methods as appropriate. Feel free to use the tools given in the example we covered. 
- Suggest: Encode the data as shown below.
***

In [ ]:
#Note: If the mlxtend library is not installed, uncomment the following line (once) and run it.
#!pip install mlxtend
import numpy as np
import pandas as pd
import mlxtend as mlx
from mlxtend.frequent_patterns import association_rules
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import fpgrowth

pd.set_option('display.max_columns', 1000) #include to avoid ... in middle of display (and use 'display(...)' when printing in cells)

ModuleNotFoundError: No module named 'mlxtend'

In [ ]:
# From the Business Problem above, discounting a product by X% causes customers to purchase the product Y% more frequently.
price_discount  = 0.10
purchase_uplift = 0.08

In [ ]:
data_file_name = 'Sp26_TEB_French_Bakery_V2.xlsx'

In [ ]:
# load the Transaction data (see the ReadMe sheet in the Excel workbook for more information)
transaction_df = pd.read_excel(data_file_name,sheet_name = 'TxToItem')
transaction_df

,TxID,ItemID,Quantity
0,150040,5,1
1,150040,86,3
2,150041,86,2
3,150041,85,1
4,150042,141,5
...,...,...,...
234000,288911,32,1
234001,288911,12,1
234002,288911,32,2
234003,288912,141,1


In [ ]:
# load the Item data
item_df = pd.read_excel(data_file_name,sheet_name = 'Items')
item_df

,ItemID,Name,UnitPrice,Category,Subcategory
0,1,Unknown,0.00,Other,Unclassified
1,2,12 Macarons,11.70,Confectionery & Snacks,Small Sweets
2,3,Armorican Cake,2.92,Pastry & Desserts,Cakes (Whole)
3,4,Article 295,0.00,Other,Unclassified
4,5,Baguette,1.05,Bread,Artisan & Daily Bread
...,...,...,...,...,...
144,145,Tropézienne,2.34,Pastry & Desserts,Individual Pastries
145,146,Raspberry Tropézienne,2.46,Pastry & Desserts,Individual Pastries
146,147,Tulip Pastry,4.10,Other,Unclassified
147,148,Vienna Bread,1.29,Bread,Artisan & Daily Bread


In [ ]:
# Check for null entries (and fix them if necessary)
transaction_df.isnull().values.any()

False

In [ ]:
transaction_df.head()

,TxID,ItemID,Quantity
0,150040,5,1
1,150040,86,3
2,150041,86,2
3,150041,85,1
4,150042,141,5


In [ ]:
# Find the unique items
items = set()
items.update(transaction_df['ItemID'].unique())
items = sorted(items)
#print(items)

In [ ]:
# Encode the Transaction data
from mlxtend.preprocessing import TransactionEncoder
from collections import defaultdict

transaction_items = defaultdict(list)
for transaction in transaction_df.values.tolist():
    transaction_items[transaction[0]].append(transaction[1])

dataset_modified = list(transaction_items.values())

te = TransactionEncoder()
te_ary = te.fit(dataset_modified).transform(dataset_modified)
te_ary

encoded_transaction_df= pd.DataFrame(te_ary, columns=te.columns_)
display(encoded_transaction_df.head(5))

,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149
0,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False
3,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,

***
# Section: 2 - Explore the Data
***

***
## Section: 2.1 - Get the Transaction and Item Sizes
- Calculate the **number_of_transactions** and **number_of_items**.
***

In [ ]:
number_of_transactions = encoded_transaction_df.shape[0]
number_of_items = encoded_transaction_df.shape[1]

transaction_item_size_summary = pd.DataFrame({
    'Measure': ['Number of transactions', 'Number of unique items'],
    'Value': [number_of_transactions, number_of_items]
})

display(transaction_item_size_summary)
print(f"number_of_transactions = {number_of_transactions:,}")
print(f"number_of_items = {number_of_items:,}")


***
## Section: 2.2 - Evaluate the Itemset and Rule Size & Complexity
- Calculate the **maximum number of Itemsets** that could be created from the items (without considering the actual transaction data). Show your work.
- Calculate the **maximum number of Rules** that can be created from the items (without considering the actual transaction data). Show your work.
- What do the calculations suggest as a **potential cause of concern**? Hint: Complexity.
- What might you do to manage these concerns?
***

In [ ]:
import math

# Maximum possible itemsets from n items, without looking at actual transaction data:
# every non-empty subset of the n items can be an itemset, so the count is 2^n - 1.
max_number_of_itemsets = (2 ** number_of_items) - 1

# Maximum possible association rules from n items:
# for each item, it can be absent, in the antecedent, or in the consequent = 3^n choices,
# then remove cases where antecedent or consequent is empty.
max_number_of_rules = (3 ** number_of_items) - (2 ** (number_of_items + 1)) + 1

# Since this homework focuses on two-item rules, these smaller counts are useful context.
max_two_itemsets = math.comb(number_of_items, 2)
max_two_item_rules = 2 * max_two_itemsets

complexity_summary = pd.DataFrame({
    'Calculation': [
        'All possible non-empty itemsets',
        'All possible association rules',
        'Possible two-itemsets only',
        'Possible two-item rules only'
    ],
    'Formula': [
        '2^n - 1',
        '3^n - 2^(n+1) + 1',
        'C(n, 2)',
        '2 * C(n, 2)'
    ],
    'Value': [
        max_number_of_itemsets,
        max_number_of_rules,
        max_two_itemsets,
        max_two_item_rules
    ]
})

display(complexity_summary)
print(f"With n = {number_of_items}, all possible itemsets = {max_number_of_itemsets:,}")
print(f"With n = {number_of_items}, all possible rules = {max_number_of_rules:,}")
print(f"For this homework, possible two-itemsets = {max_two_itemsets:,}")
print(f"For this homework, possible two-item rules = {max_two_item_rules:,}")


**Discussion:**

The calculations show that association analysis can become extremely large very quickly. With 149 unique items, the number of possible itemsets and possible rules grows exponentially. This is a major complexity concern because evaluating every possible itemset and rule would take a large amount of time and memory, and many of those rules would probably be rare or not useful for the business problem. To manage this, I would use constraints such as minimum support, maximum itemset length, and a focus on the top two-itemsets. I would also remove or carefully handle non-product entries such as “Unknown” or “Slice” when making the final business recommendation, because discounting a fee or unclear item would not be a realistic marketing action.


***
# Section: 3 - Itemset Generation
- Create a set of 20 two-item sets with highest support. Sort them in decreasing order of support.
- NOTE: You can set the **apriori** function to create itemsets of **max_len=2** and try various values for **min_support** to get the right number of 2-itemsets.  Then you can filter them for only the 2-itemsets.  But keep the 1and2-itemsets and the 1-itemsets around - they might be useful later. 
- Show the results, briefly.
- Explain what you did and why you did it.
***

In [ ]:
# Create 1-itemsets and 2-itemsets using Apriori.
# I use a low minimum support so that the most important two-itemsets are included,
# then I explicitly select the top 20 two-itemsets by support.
min_support_threshold = 0.001

one_and_two_itemsets = apriori(
    encoded_transaction_df,
    min_support=min_support_threshold,
    use_colnames=True,
    max_len=2
)

one_itemsets = one_and_two_itemsets[
    one_and_two_itemsets['itemsets'].apply(lambda x: len(x) == 1)
].copy()

two_itemsets = one_and_two_itemsets[
    one_and_two_itemsets['itemsets'].apply(lambda x: len(x) == 2)
].copy()

top_20_two_itemsets = two_itemsets.sort_values(
    by='support',
    ascending=False
).head(20).reset_index(drop=True)

item_name_lookup = item_df.set_index('ItemID')['Name'].to_dict()

def itemset_to_names(itemset):
    return ', '.join(item_name_lookup.get(int(item), str(item)) for item in sorted(itemset))

top_20_two_itemsets_display = top_20_two_itemsets.copy()
top_20_two_itemsets_display['Item Names'] = top_20_two_itemsets_display['itemsets'].apply(itemset_to_names)
top_20_two_itemsets_display['Support %'] = top_20_two_itemsets_display['support'] * 100

display(top_20_two_itemsets_display[['support', 'Support %', 'itemsets', 'Item Names']])

print("I used Apriori with max_len=2 so the result stays focused on one-itemsets and two-itemsets.")
print("Then I sorted the two-itemsets by support and kept the top 20, because the homework asks for the 20 two-itemsets with highest support.")


***
# Section: 4 - Generate Rules
- For the two-itemsets created above, create the related rules.
- Use the **association_rules** function.  You need to pass in the 1and2_itemsets to the function.  If you set **min_threshold=0.0**, you will get all of the rules.
***

In [ ]:
# Generate rules only for the 20 two-itemsets selected above.
# The association_rules function needs the one-item supports as well as the two-item supports.
selected_pair_itemsets = set(top_20_two_itemsets['itemsets'])
selected_itemsets_for_rules = pd.concat(
    [one_itemsets, top_20_two_itemsets],
    ignore_index=True
)

try:
    rules = association_rules(
        selected_itemsets_for_rules,
        metric='confidence',
        min_threshold=0.0,
        num_itemsets=len(encoded_transaction_df)
    )
except TypeError:
    # Older mlxtend versions did not require the num_itemsets argument.
    rules = association_rules(
        selected_itemsets_for_rules,
        metric='confidence',
        min_threshold=0.0
    )

rules = rules[
    rules.apply(
        lambda row: (row['antecedents'].union(row['consequents']) in selected_pair_itemsets)
        and (len(row['antecedents']) == 1)
        and (len(row['consequents']) == 1),
        axis=1
    )
].copy()

rules['Antecedent ItemID'] = rules['antecedents'].apply(lambda x: int(next(iter(x))))
rules['Consequent ItemID'] = rules['consequents'].apply(lambda x: int(next(iter(x))))
rules['Antecedent Item'] = rules['Antecedent ItemID'].map(item_name_lookup)
rules['Consequent Item'] = rules['Consequent ItemID'].map(item_name_lookup)
rules['Support %'] = rules['support'] * 100
rules['Confidence %'] = rules['confidence'] * 100

rules = rules.sort_values(
    by=['support', 'confidence', 'lift'],
    ascending=[False, False, False]
).reset_index(drop=True)

display(rules[[
    'Antecedent ItemID', 'Antecedent Item',
    'Consequent ItemID', 'Consequent Item',
    'support', 'Support %', 'confidence', 'Confidence %', 'lift'
]])


***
# Section: 5 - Rule Evaluation
- For the rules created above, find the single Item (that would be given the discount) that would cause the greatest increase in monthly store revenue.
  - This is based on the Business Problem stated at the top of this notebook.
  - Consider:
    - How much will the store's monthly revenue decrease (or increase) due to the change in price for the chosen Item (and its increased sales)?
    - How much will the store's monthly revenue increase (or decrease) due to the increased sales of the associated Items?
***

In [ ]:
# Evaluate which single item should receive the discount.
# For each rule A -> B, I treat A as the discounted item and B as the associated item.
# The dataset is treated as one month of transactions, and quantities are treated as presence/absence.

item_info = item_df.set_index('ItemID')[['Name', 'UnitPrice', 'Category', 'Subcategory']]

rules_revenue = rules.copy()
rules_revenue['Discount_ItemID'] = rules_revenue['Antecedent ItemID']
rules_revenue['Associated_ItemID'] = rules_revenue['Consequent ItemID']
rules_revenue['Discount_Item'] = rules_revenue['Discount_ItemID'].map(item_info['Name'])
rules_revenue['Associated_Item'] = rules_revenue['Associated_ItemID'].map(item_info['Name'])
rules_revenue['Discount_Item_Price'] = rules_revenue['Discount_ItemID'].map(item_info['UnitPrice'])
rules_revenue['Associated_Item_Price'] = rules_revenue['Associated_ItemID'].map(item_info['UnitPrice'])

# A few rows in the workbook are not realistic discount candidates.
# I exclude them from the final business recommendation, but keep the calculation transparent.
not_real_discount_items = {'Unknown', 'Slice', 'Article 295'}
rules_revenue['Eligible_Discount_Item'] = (
    (rules_revenue['Discount_Item_Price'] > 0)
    & (~rules_revenue['Discount_Item'].isin(not_real_discount_items))
)
rules_revenue['Eligible_Associated_Item'] = (
    (rules_revenue['Associated_Item_Price'] > 0)
    & (~rules_revenue['Associated_Item'].isin(not_real_discount_items))
)

# Revenue effect for discounted item A:
# New revenue from A = old count(A) * 1.08 * old price(A) * 0.90.
# Net effect for A = new revenue - old revenue.
rules_revenue['Discount_Item_Monthly_Count'] = rules_revenue['antecedent support'] * number_of_transactions
rules_revenue['Associated_Pair_Monthly_Count'] = rules_revenue['support'] * number_of_transactions

rules_revenue['Discount_Item_Revenue_Change'] = (
    rules_revenue['Discount_Item_Monthly_Count']
    * rules_revenue['Discount_Item_Price']
    * ((1 - price_discount) * (1 + purchase_uplift) - 1)
)

# Revenue effect for associated item B:
# Extra purchases of A = count(A) * 0.08.
# Estimated extra purchases of B = extra purchases of A * confidence(A -> B).
rules_revenue['Associated_Item_Revenue_Gain'] = (
    rules_revenue['Discount_Item_Monthly_Count']
    * purchase_uplift
    * rules_revenue['confidence']
    * rules_revenue['Associated_Item_Price']
)

rules_revenue['Estimated_Net_Revenue_Change'] = (
    rules_revenue['Discount_Item_Revenue_Change']
    + rules_revenue['Associated_Item_Revenue_Gain']
)

eligible_revenue_rules = rules_revenue[
    rules_revenue['Eligible_Discount_Item'] & rules_revenue['Eligible_Associated_Item']
].copy()

revenue_results = eligible_revenue_rules.sort_values(
    by='Estimated_Net_Revenue_Change',
    ascending=False
).reset_index(drop=True)

revenue_display_columns = [
    'Discount_ItemID', 'Discount_Item',
    'Associated_ItemID', 'Associated_Item',
    'support', 'confidence', 'lift',
    'Discount_Item_Price', 'Associated_Item_Price',
    'Discount_Item_Revenue_Change',
    'Associated_Item_Revenue_Gain',
    'Estimated_Net_Revenue_Change'
]

display(revenue_results[revenue_display_columns])

best_rule = revenue_results.iloc[0]
print("Best recommended discount item:", best_rule['Discount_Item'])
print("Associated item:", best_rule['Associated_Item'])
print(f"Estimated net monthly revenue change: ${best_rule['Estimated_Net_Revenue_Change']:,.2f}")


**Discussion:**

Based on the revenue calculation, the best realistic discount choice is **Croissant**, using the rule **Croissant → Chocolate Croissant**. The estimated net monthly revenue increase is positive because the extra expected revenue from Chocolate Croissant purchases is larger than the revenue loss from discounting Croissants. I excluded unclear or non-product items such as “Unknown,” “Article 295,” and “Slice” from the final recommendation because discounting a fee or unclear record would not be a practical business promotion. This also makes the final answer more useful for the bakery, since the store would want to discount an actual product that customers can intentionally buy.


***
# Section: 7 - Conclusions
- Write a paragraph on what you discovered or learned from this homework.
***

In this homework, I learned how association analysis can be used to connect a technical data-mining result to a practical business decision. The Apriori algorithm helped identify common two-item combinations in the bakery transactions, and the association rules showed that support alone is not enough because rule direction, confidence, lift, item price, and business assumptions all matter. I also learned that preprocessing decisions are important. Items such as “Slice” appeared very frequently, but they are not really a normal product to discount, so blindly choosing the highest-support rule could lead to a poor recommendation. Overall, the best realistic recommendation from my analysis is to discount Croissants because the rule connecting Croissants to Chocolate Croissants gives the strongest positive estimated monthly revenue effect among the eligible product rules.


***
### END-OF-SUBMISSION
***